# 02 - EDA de los informes radiologicos

Objetivo (plan Fase 2): vocabulario, longitud, hallazgos mencionados, idiomas presentes (hasta 9, sin columna de idioma - ver src/config.py::REPORT_LANGUAGE_COUNT). Confirmar que test.csv NO tiene columna Report (ver src/labelers.py docstring).

In [ ]:
# Corre en Kaggle Notebook (contra el dataset completo montado) o en
# local (contra data/raw/, tras 00_export_local_gold_subset.ipynb -
# ver bloque RAW_DIR mas abajo). Sync notebook<->Kaggle es manual por
# ahora, asi que las constantes de src/config.py se repiten aqui a
# mano; mantenerlas sincronizadas si config.py cambia.

import random
import re
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

# Kaggle si el dataset esta montado ahi; si no, local (data/raw/, tras
# extraer ahi el zip de notebooks/00_export_local_gold_subset.ipynb -
# ver ese notebook). En local solo estara el subset gold de DICOM, pero
# los 4 CSVs (incluido train.csv con Report para las 4,407 filas) son
# completos, asi que este notebook corre igual de bien en cualquiera.
_KAGGLE_RAW = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
_LOCAL_RAW = Path("../data/raw")  # relativo a notebooks/
RAW_DIR = _KAGGLE_RAW if _KAGGLE_RAW.exists() else _LOCAL_RAW
assert RAW_DIR.exists(), (
    f"Dataset no encontrado ni en {_KAGGLE_RAW} ni en {_LOCAL_RAW} - "
    "corre 00_export_local_gold_subset.ipynb en Kaggle primero, o anade "
    "el dataset como input del kernel."
)
print("Corriendo contra:", RAW_DIR)

# Copiado de src/config.py::OFFICIAL_LABEL_COLUMNS.
OFFICIAL_LABEL_COLUMNS = {
    "acl_injury": "ACL",
    "mcl_injury": "MCL",
    "medial_meniscus_tear": "Medial Meniscus",
    "lateral_meniscus_tear": "Lateral Meniscus",
    "oa_medial_compartment": "Medial OA",
    "oa_lateral_compartment": "Lateral OA",
    "oa_patellofemoral_compartment": "PF OA",
    "effusion": "Effusion",
    "synovitis": "Synovitis",
    "bakers_cyst": "Baker's",
    "bone_contusion": "Contusion",
    "fracture": "Fracture",
}
LABEL_COLS = list(OFFICIAL_LABEL_COLUMNS.values())
REPORT_LANGUAGE_COUNT = 9  # de config.py - hipotesis a contrastar en la seccion D.

random.seed(42)
np.random.seed(42)

## A. Carga - confirmar ausencia de `Report` en `test.csv`, separar gold vs. weak

Re-confirma lo visto en `01_eda_dicom.ipynb` seccion B (barato, y este notebook no depende de haber corrido ese antes). Ademas separa el subset gold (58 filas, las 12 etiquetas pobladas) del resto ("weak", solo-reporte) - Fase 3 va a validar `label_report()` contra el gold, asi que si el gold no es representativo del resto en longitud/idioma/vocabulario, esa validacion no generaliza al conjunto donde de verdad hace falta el weak labeling.

In [ ]:
train = pd.read_csv(RAW_DIR / "train.csv")
test = pd.read_csv(RAW_DIR / "test.csv")
assert "Report" not in test.columns, "Report aparece en test.csv - contradice 01-B."
print("OK: Report ausente en test.csv (re-confirmado)")

n_labels_present = train[LABEL_COLS].notna().sum(axis=1)
gold_mask = n_labels_present == len(LABEL_COLS)
gold = train.loc[gold_mask].copy()
weak = train.loc[~gold_mask].copy()
assert gold["Report"].notna().all() and weak["Report"].notna().all()
print(f"gold: {len(gold)} filas | weak (solo-reporte): {len(weak)} filas")

## B. Longitud de los informes - gold vs. weak

Caracteres, palabras y lineas por informe. Si el gold es sistematicamente mas largo/corto o tiene una distribucion de longitud distinta al weak, es una senal de que no es una muestra representativa (importa para el diseno de la validacion en Fase 3).

In [ ]:
def length_stats(df, name):
    n_chars = df["Report"].str.len()
    n_words = df["Report"].str.split().str.len()
    n_lines = df["Report"].str.count("\n") + 1
    stats = pd.DataFrame({"n_chars": n_chars, "n_words": n_words, "n_lines": n_lines})
    print(f"--- {name} (n={len(df)}) ---")
    print(stats.describe())
    print()
    return stats

gold_stats = length_stats(gold, "gold")
weak_stats = length_stats(weak, "weak")

## C. Deteccion de hard-wrap

El notebook de referencia prvsiyan/rsna-knee-read-the-report-then-the-knee afirma que una parte grande del corpus llega hard-wrapped a una columna fija, es decir, frases cortadas a mitad por un salto de linea sin puntuacion de cierre - y que por eso su funcion `unwrap()` (que junta la linea siguiente si la anterior no termina en puntuacion de cierre y la siguiente no empieza en mayuscula) es necesaria antes de segmentar en clausulas. No dar esto por bueno solo porque lo dice el notebook de referencia: medirlo contra nuestros propios datos, igual que se hizo con `Fluid_Sensitive`/`Fat_Suppression` en Fase 1 (donde el aviso oficial resulto ser demasiado cauto).

In [ ]:
def frac_wrap_candidate_lines(text):
    """Fraccion de saltos de linea consecutivos que parecen wrap, no fin de frase."""
    if not isinstance(text, str):
        return np.nan
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    if len(lines) < 2:
        return 0.0
    candidates = 0
    for a, b in zip(lines, lines[1:]):
        if (not re.search(r"[.;:!?>*\u2022]$", a)
                and len(a.split()) >= 4
                and b[:1] and not b[:1].isupper()):
            candidates += 1
    return candidates / (len(lines) - 1)

train["frac_wrap_candidates"] = train["Report"].apply(frac_wrap_candidate_lines)
print(train["frac_wrap_candidates"].describe())
print(f"\nInformes con >=1 linea candidata a hard-wrap: {(train['frac_wrap_candidates'] > 0).mean():.1%}")

has_wrap = train.loc[train["frac_wrap_candidates"] > 0.3, "Report"]
if len(has_wrap) > 0:
    print("\nEjemplo (frac_wrap_candidates > 0.3):\n")
    print(has_wrap.iloc[0][:600])
else:
    print("\nNingun informe supera el umbral de 0.3 - revisar si el fenomeno existe en esta muestra.")

## D. Idiomas presentes - sin columna de idioma

`REPORT_LANGUAGE_COUNT = 9` en `src/config.py` viene de los dos notebooks de referencia, no de una medicion propia - contrastarlo aqui. Dos capas: (1) script dominante (Latino/Griego/Cirilico/...) por rango Unicode, que no depende de ninguna libreria y por tanto siempre funciona; (2) dentro del script latino, un intento con `langdetect` si esta disponible en la imagen del kernel (puede no estarlo - no forma parte de los requirements de este proyecto, es solo para orientar, no para decidir nada de `src/`).

In [ ]:
def dominant_script(text):
    if not isinstance(text, str) or not text.strip():
        return "empty"
    counts = Counter()
    for ch in text:
        if ch.isspace() or ch.isdigit() or not ch.isalpha():
            continue
        try:
            name = unicodedata.name(ch)
        except ValueError:
            continue
        counts[name.split()[0]] += 1  # "LATIN", "GREEK", "CYRILLIC", ...
    if not counts:
        return "unknown"
    return counts.most_common(1)[0][0]

train["script"] = train["Report"].apply(dominant_script)
print("Script dominante por informe:")
print(train["script"].value_counts())

In [ ]:
try:
    from langdetect import DetectorFactory, detect
    DetectorFactory.seed = 42
    HAS_LANGDETECT = True
except ImportError:
    HAS_LANGDETECT = False
    print("langdetect no disponible en este kernel - solo se reporta script (celda anterior), no idioma exacto.")

if HAS_LANGDETECT:
    latin_mask = train["script"] == "LATIN"
    sample_n = min(500, int(latin_mask.sum()))
    sample = train.loc[latin_mask, "Report"].sample(n=sample_n, random_state=42)

    def safe_detect(t):
        try:
            return detect(t)
        except Exception:
            return "unknown"

    langs = sample.apply(safe_detect)
    print(f"Idiomas detectados en una muestra de {sample_n} informes de script latino:")
    print(langs.value_counts())
    n_non_latin_scripts = train.loc[~latin_mask, "script"].nunique()
    print(f"\nIdiomas latinos distintos: {langs.nunique()} + {n_non_latin_scripts} script(s) no latinos"
          f" = comparar contra REPORT_LANGUAGE_COUNT={REPORT_LANGUAGE_COUNT}")

## E. Informes duplicados / plantillas compartidas

Ambos notebooks de referencia avisan de esto como un leak de validacion: informes byte-identicos entre estudios (p.ej. una plantilla para rodilla sin hallazgos), por lo que separar ese grupo entre train/val de un fold puntua el modelo contra un target que ya vio. `src/labelers.py::report_group_key()` (Fase 5) existe para esto. Aqui solo se mide el tamano real del problema en nuestros datos - cuantos grupos hay, que tan grandes son, y si el gold comparte plantilla con el weak (si comparte, agrupar por hash es obligatorio tambien para no filtrar gold<->weak, no solo weak<->weak).

In [ ]:
def normalize_for_hash(text):
    if not isinstance(text, str):
        return ""
    t = unicodedata.normalize("NFKD", text.lower())
    t = "".join(ch for ch in t if not unicodedata.combining(ch))
    return re.sub(r"\s+", " ", t).strip()

train["report_norm"] = train["Report"].apply(normalize_for_hash)
dup_group_sizes = train.groupby("report_norm").size().sort_values(ascending=False)
dup_groups = dup_group_sizes[dup_group_sizes > 1]
print(f"Grupos de informes duplicados (texto identico tras normalizar): {len(dup_groups)}")
print(f"Estudios en algun grupo duplicado: {dup_groups.sum()} / {len(train)} ({dup_groups.sum() / len(train):.1%})")
print("\nTop 10 grupos mas grandes:")
print(dup_group_sizes.head(10))

if len(dup_groups) > 0:
    biggest = dup_group_sizes.index[0]
    print("\nEjemplo de la plantilla mas repetida:\n")
    print(biggest[:400])

gold_norm = set(train.loc[gold_mask, "report_norm"])
weak_norm = set(train.loc[~gold_mask, "report_norm"])
overlap = gold_norm & weak_norm
print(f"\nPlantillas compartidas entre gold y weak: {len(overlap)}")

## F. Vocabulario

Frecuencia de tokens tras normalizar (mismo `normalize_for_hash` de la seccion E, tokenizado con un regex que cubre latino + griego + cirilico). Sirve como insumo para disenar las labeling functions de la Fase 3 - **no** se disena el labeler aqui, eso es scope de `03_labeler_validation.ipynb`.

In [ ]:
_TOKEN_RE = re.compile(r"[a-z\u00C0-\u017F\u0370-\u03FF\u0400-\u04FF]+")

def tokenize(norm_text):
    return _TOKEN_RE.findall(norm_text)

all_tokens = Counter()
for t in train["report_norm"]:
    all_tokens.update(tokenize(t))

print(f"Vocabulario: {len(all_tokens)} tokens distintos, {sum(all_tokens.values())} tokens totales")
print("\nTop 40 tokens mas frecuentes:")
for tok, n in all_tokens.most_common(40):
    print(f"  {tok}: {n}")

## Resumen - pegar resultados aqui tras correr en Kaggle

(Igual que en `01_eda_dicom.ipynb`: correr este notebook en Kaggle contra el dataset montado, pegar los numeros/hallazgos reales de vuelta en la conversacion para actualizar `README.md` Progress y `RESOURCES.md` antes de pasar a la Fase 3.)

Preguntas que este notebook deberia dejar contestadas:
- Longitud de gold vs. weak: comparable o hay que tenerlo en cuenta al validar Fase 3?
- El hard-wrap del notebook de referencia se observa aqui, y en que proporcion?
- Cuantos idiomas/scripts distintos hay realmente? Coincide con REPORT_LANGUAGE_COUNT=9?
- Cuantos estudios caen en grupos de informes duplicados, y se solapan gold/weak?
- Que tokens dominan el vocabulario y encajan con los 12 hallazgos?